# Сравнение LogReg и Random Forest

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report
from features import build_features

df = pd.read_csv('dataset.csv')
df = build_features(df)
df = df.dropna(subset=['дата_публикации']).sort_values('дата_публикации').reset_index(drop=True)
df.shape

(2117, 38)

In [ ]:
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

print(f"Train: {len(train)} строк, позитивов {train['label'].sum()}")
print(f"Test: {len(test)} строк, позитивов {test['label'].sum()}")
print(f"Период train: {train['дата_публикации'].min()} - {train['дата_публикации'].max()}")
print(f"Период test: {test['дата_публикации'].min()} - {test['дата_публикации'].max()}")

Train: 1693 строк, позитивов 117
Test:  424 строк, позитивов 50
Период train: 2026-06-12 09:45:57 - 2026-07-07 15:11:11
Период test:  2026-07-07 15:16:00 - 2026-07-14 20:52:00


In [ ]:
tfidf = TfidfVectorizer(max_features=200)
X_train_text = tfidf.fit_transform(train['название_стем'])
X_test_text = tfidf.transform(test['название_стем'])

cat_cols = ['способ_группа', 'тип_торгов']
ohe = OneHotEncoder(handle_unknown='ignore')
X_train_cat = ohe.fit_transform(train[cat_cols])
X_test_cat = ohe.transform(test[cat_cols])

num_cols = ['москва', 'нмц_указана', 'лог_цена', 'дни_до_дедлайна']
X_train_num = csr_matrix(train[num_cols].values)
X_test_num = csr_matrix(test[num_cols].values)

X_train = hstack([X_train_text, X_train_cat, X_train_num])
X_test = hstack([X_test_text, X_test_cat, X_test_num])

y_train = train['label']
y_test = test['label']

feature_names = list(tfidf.get_feature_names_out()) + list(ohe.get_feature_names_out(cat_cols)) + num_cols
len(feature_names)

213

## LogReg

In [ ]:
logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(X_train, y_train)

proba_lr = logreg.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, proba_lr)
print(f"LogReg ROC AUC: {auc_lr:.4f}")

precision_lr, recall_lr, thresholds_lr = precision_recall_curve(y_test, proba_lr)
target_recall = 0.85
idx_lr = np.where(recall_lr[:-1] >= target_recall)[0]
best_lr = idx_lr[np.argmax(precision_lr[idx_lr])] if len(idx_lr) else np.argmax(recall_lr[:-1])
threshold_lr = thresholds_lr[best_lr]
print(f"Порог: {threshold_lr:.3f}, precision={precision_lr[best_lr]:.3f}, recall={recall_lr[best_lr]:.3f}")

y_pred_lr = (proba_lr >= threshold_lr).astype(int)
print(classification_report(y_test, y_pred_lr, target_names=['не интересно', 'интересно']))

LogReg ROC AUC: 0.9009
Порог: 0.451, precision=0.405, recall=0.900
              precision    recall  f1-score   support

не интересно       0.98      0.82      0.90       374
   интересно       0.41      0.90      0.56        50

    accuracy                           0.83       424
   macro avg       0.69      0.86      0.73       424
weighted avg       0.92      0.83      0.86       424



## Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

proba_rf = rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, proba_rf)
print(f"Random Forest ROC AUC: {auc_rf:.4f}")

precision_rf, recall_rf, thresholds_rf = precision_recall_curve(y_test, proba_rf)
idx_rf = np.where(recall_rf[:-1] >= target_recall)[0]
best_rf = idx_rf[np.argmax(precision_rf[idx_rf])] if len(idx_rf) else np.argmax(recall_rf[:-1])
threshold_rf = thresholds_rf[best_rf]
print(f"Порог: {threshold_rf:.3f}, precision={precision_rf[best_rf]:.3f}, recall={recall_rf[best_rf]:.3f}")

y_pred_rf = (proba_rf >= threshold_rf).astype(int)
print(classification_report(y_test, y_pred_rf, target_names=['не интересно', 'интересно']))

Random Forest ROC AUC: 0.9080
Порог: 0.107, precision=0.411, recall=0.920
              precision    recall  f1-score   support

не интересно       0.99      0.82      0.90       374
   интересно       0.41      0.92      0.57        50

    accuracy                           0.83       424
   macro avg       0.70      0.87      0.73       424
weighted avg       0.92      0.83      0.86       424



## Сравнение

In [ ]:
summary = pd.DataFrame({
    'модель': ['LogReg', 'Random Forest'],
    'ROC AUC': [auc_lr, auc_rf],
    'порог': [threshold_lr, threshold_rf],
    'precision': [precision_lr[best_lr], precision_rf[best_rf]],
    'recall': [recall_lr[best_lr], recall_rf[best_rf]],
})
summary

,модель,ROC AUC,порог,precision,recall
0,LogReg,0.900909,0.450569,0.405405,0.90
1,Random Forest,0.907968,0.106667,0.410714,0.92
